# PISCES — chlorophyll seasonal cycle off the Iberian upwelling

Demonstrates the **multi-variable subset pattern**: pull two PISCES
biogeochem variables (`chl` chlorophyll-a, `o2` dissolved oxygen) over a
coastal box off Iberia for a multi-year window, group by month, and plot
the seasonal cycle.

PISCES is `cmems_mod_glo_bgc_my_0.25deg_P1D-m` — daily, 1/4° grid,
global. Variables are 4-D `(time, depth, lat, lon)`; we clip to the
surface 0-10 m to keep the request small.

In [ ]:
import os
from pathlib import Path

import numpy as np

from earthlens import EarthLens
from earthlens.cmems import Catalog
from pyramids.netcdf import NetCDF

OUT_DIR = Path('data/cmems-pisces')
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_ID = 'cmems_mod_glo_bgc_my_0.25deg_P1D-m'
VARIABLES = ['chl', 'o2']
BBOX = dict(lat_lim=[40.0, 44.0], lon_lim=[-12.0, -8.0])   # Iberian upwelling box

ds = Catalog().get_dataset(DATASET_ID)
print(f'{DATASET_ID}: cadence={ds.cadence}, domain={ds.domain}')
for v in VARIABLES:
    print(f'  {v}: units={ds.variables[v].units!r}, long_name={ds.variables[v].long_name!r}')

## Download — three years of daily PISCES at the surface

Multi-year × multi-variable × surface-clipped, in one server-side
subset call.

In [ ]:
earthlens = EarthLens(
    data_source='cmems',
    start='2018-01-01',
    end='2020-12-31',
    temporal_resolution='daily',
    variables={DATASET_ID: VARIABLES},
    **BBOX,
    path=str(OUT_DIR),
    minimum_depth=0.0,
    maximum_depth=10.0,
    service_username=os.environ.get('COPERNICUSMARINE_SERVICE_USERNAME'),
    service_password=os.environ.get('COPERNICUSMARINE_SERVICE_PASSWORD'),
)
paths = earthlens.download()
print(paths)

## Compute the box-averaged monthly climatology for each variable

Average across `(depth, lat, lon)` per day, then group by calendar month
across the three years. Both arrays end up as 12-element vectors keyed on
month-of-year (1-12).

In [ ]:
import datetime as dt

nc = NetCDF.read_file(str(paths[0]), read_only=True)
time = nc.read_array('time')           # toolbox writes 'seconds since 1970-01-01'
time_units = getattr(nc.meta_data.variables['time'], 'unit', '')
epoch = dt.datetime(1970, 1, 1)
dates = [epoch + dt.timedelta(seconds=float(t)) for t in time]
months = np.array([d.month for d in dates])

climatology = {}
for v in VARIABLES:
    arr = nc.read_array(v)                # (time, depth, lat, lon)
    daily_mean = arr.mean(axis=(1, 2, 3))  # (time,)
    climatology[v] = np.array([daily_mean[months == m].mean() for m in range(1, 13)])
    print(f'{v}: {climatology[v]}')
nc.close()

## Plot the seasonal cycle

Iberian upwelling chlorophyll peaks in spring / late summer with the
wind-driven upwelling events; dissolved oxygen tracks the same pattern
inversely (warmer water holds less O2).

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
months = np.arange(1, 13)
ax1.bar(months, climatology['chl'], color='tab:green')
ax1.set_xlabel('Month')
ax1.set_ylabel('chl (mg m-3)')
ax1.set_title('PISCES chlorophyll-a climatology')
ax1.set_xticks(months)
ax2.bar(months, climatology['o2'], color='tab:blue')
ax2.set_xlabel('Month')
ax2.set_ylabel('o2 (mmol m-3)')
ax2.set_title('PISCES dissolved oxygen climatology')
ax2.set_xticks(months)
fig.suptitle('Iberian upwelling box, 2018-2020 surface (0-10 m) mean')
fig.tight_layout()